In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score

from imblearn.over_sampling import SMOTE
from imblearn.over_sampling import RandomOverSampler

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier


In [ ]:
train_df = pd.read_csv('/Users/soleymani/Downloads/Training Data.csv')
test_df = pd.read_csv('/Users/soleymani/Downloads/Test Data.csv')

In [ ]:
train_df.head()

In [ ]:
train_df.info()
train_df.describe()

In [ ]:
train_df.shape

In [ ]:
train_df.isnull().sum()

In [ ]:
print("Number of duplicates:", train_df.duplicated().sum())

In [ ]:
num_cols = ["Income", "Age", "Experience", "CURRENT_JOB_YRS", "CURRENT_HOUSE_YRS"]
for col in num_cols:
    train_df = train_df[train_df[col] >= 0]

In [ ]:
train_df.info()

In [ ]:
sns.countplot(x='Risk_Flag', data=train_df)
plt.title('Target Distribution')
plt.show()

In [ ]:
for num_col in num_cols:
    train_df[[num_col]].hist(bins=30, figsize=(10,5))
    plt.tight_layout()
    plt.show()

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(train_df[num_cols + ['Risk_Flag']].corr(),annot=True, fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
cat_cols = ['Married/Single', 'House_Ownership', 'Car_Ownership', 'Profession', 'CITY', 'STATE']

for col in cat_cols:
    print(f'\n{col}')
    print(train_df[col].value_counts().head(10))

In [ ]:
for col in ['Married/Single', 'House_Ownership', 'Car_Ownership']:
    plt.figure(figsize=(6, 4))
    sns.countplot(x=col, data=train_df)
    plt.title(col)
    plt.xticks(rotation=45)
    plt.show()

In [ ]:
for col in train_df[cat_cols].columns:
    print(col , train_df[col].nunique())

# uniquie in risk
print("Risk_Flag", train_df["Risk_Flag"].nunique())

In [ ]:
train_df = train_df.copy()

In [ ]:
train_df["Income_Per_Experience"] = train_df["Income"] / (train_df["Experience"] + 1).astype(float)
train_df["Job_Stability_Index"] = train_df["CURRENT_JOB_YRS"] / (train_df["Age"] + 1).astype(float)
train_df["Financial_Strength"] = (
    (train_df["House_Ownership"] == "owned").astype(int) +
    (train_df["Car_Ownership"] == "yes").astype(int)
)

In [ ]:
train_df = train_df.drop(columns = ['Id'])
test_df = test_df.drop(columns = ['ID'])

In [ ]:
le = LabelEncoder()
for col in cat_cols:
    train_df[col] = le.fit_transform(train_df[col])

In [ ]:
x_train = train_df.drop(columns=["Risk_Flag"])
y_train = train_df["Risk_Flag"]

In [ ]:
ros = RandomOverSampler()
x_train, y_train = ros.fit_resample(x_train,y_train)
y_train.value_counts()

In [ ]:
x_train_tr, x_valid, y_train_tr, y_valid = train_test_split(x_train, y_train, test_size=0.2, random_state=42, stratify=y_train)

In [ ]:
x_train_tr.head()

In [ ]:
x_valid.head()

In [ ]:
# y_train_tr =pd.DataFrame(y_train_tr)
y_train_tr.head()

In [ ]:
# y_valid =pd.DataFrame(y_valid)
y_valid.head()

In [ ]:
scaler = StandardScaler()
x_train_tr = scaler.fit_transform(x_train_tr[num_cols])
x_valid = scaler.transform(x_valid[num_cols])

In [ ]:
smote = SMOTE(random_state=42)
x_train_tr, y_train_tr = smote.fit_resample(x_train_tr, y_train_tr)

In [ ]:
xgb = XGBClassifier(base_score=0.5, booster='gbtree', colsample_bylevel=1,
              colsample_bynode=1, colsample_bytree=1, gamma=0, gpu_id=-1,
              importance_type='gain', interaction_constraints='',
              learning_rate=0.3, max_delta_step=0, max_depth=9,
              min_child_weight=1, missing=np.nan, monotone_constraints='()',
              n_estimators=1000, n_jobs=8, num_parallel_tree=1, random_state=0,
              reg_alpha=0, reg_lambda=1, scale_pos_weight=1, subsample=1,
              tree_method='exact', validate_parameters=1, verbosity=None)

In [ ]:
xgb.fit(x_train_tr,y_train_tr)
predict_train = xgb.predict(x_train_tr)
predict_valid = xgb.predict(x_valid)

In [ ]:
print("Train Score:", roc_auc_score(predict_train, y_train_tr))
print("Train error:", mean_squared_error(predict_train,y_train_tr))
print("Test Score:", roc_auc_score(predict_valid, y_valid))
print("Test error:", mean_squared_error(predict_valid,y_valid))

In [ ]:
def plot(y_train, pred, title=None):
    
    matrix = confusion_matrix(y_train, pred)
    plt.figure(figsize = (5,5))
    sns.heatmap(data= matrix, annot=True, cmap='Blues', fmt='g')
    
    plt.xlabel("Actual")
    plt.ylabel("Predicted")
    plt.title(title)
    plt.show

    plot(y_train_tr,predict_train, "Training Predictions")

    plot(y_valid,predict_test, "Test Predictions")

---

In [ ]:
# model = RandomForestClassifier(
#     n_estimators=200,
#     random_state=42,
#     class_weight='balanced'
# )
# model.fit(x_train_tr, y_train_tr)
#
# y_pred = model.predict(x_valid)
#
# print(accuracy_score(y_valid, y_pred))
# print(classification_report(y_valid, y_pred))
#
# y_proba = model.predict_proba(x_valid)[:, 1]
# threshold = 0.45
# y_pred = (y_proba >= threshold).astype(int)
# print(confusion_matrix(y_valid, y_pred))
# print(roc_auc_score(y_valid, y_proba))
#

---

In [ ]:
 # model = DecisionTreeClassifier(criterion="entropy",random_state=42)
# model.fit(x_train_tr, y_train_tr)
# y_pred = model.predict(x_valid)
# accuracy = model.score(x_valid, y_valid)
# y_proba = model.predict_proba(x_valid)[:, 1]
# threshold = 0.45
# y_pred = (y_proba >= threshold).astype(int)
# print(classification_report(y_valid, y_pred))
# print(confusion_matrix(y_valid, y_pred))
# print(roc_auc_score(y_valid, y_proba))

 ---